# README

- **Author**: `黃書佑`
- **Created At**: `2026-05-11`
- **Last Modified At**: `2026-05-11`

---

## What does this file do?

- `Data Preprocessing`
- `Data Cleaning`
- `Merge data`

---

## What does this file take?

- **Source Data Sets**:  
  1. `/data/raw/members.csv` 
    - Description: `Member ID, Name, Gender, Registration Date, Birthday, Membership Tier (Silver, Gold, Diamond), Coupon Type` 
  2. `/data/raw/sales-2024.csv`
    - Description: `Member ID, Transaction ID, Purchase Date, Payment Method, Product Item, Quantity, Size, Unit Price, Total price` 

  
---

## What does this file output?

- `/data/clean/members.csv`  
  - Description: `Member ID, Name, Gender, Registration Date, Birthday, Membership Tier (Silver, Gold, Diamond), Age, Register Year`
- `/data/clean/sales.csv`
  - Description: `Member ID, Transaction ID, Purchase Date, Payment Method, Product Item, Size, Price`
- `/data/clean/merge.csv`
  - Description: `Member ID, Transaction ID, Purchase Date, Payment Method, Product Item, Size, Price, Name, Gender, Registration Date, Birthday, Membership Tier (Silver, Gold, Diamond), Age, Register Year`

In [2]:
# for data cleaning and processing
import numpy as np
import pandas as pd

## Sales data

In [4]:
sales = pd.read_csv('../raw/sales-2024.csv')
sales.head()

,transaction_id,member_id,purchase_datetime,payment_method,category,product,size,quantity,unit_price,total_price
0,T00000001,3428,2024-01-01 08:00:01,現金,調味茶,櫻桃紅茶,L,2,120,240
1,T00000002,3428,2024-01-01 08:00:01,電子支付,調味茶,櫻桃紅茶,L,1,120,120
2,T00000003,3428,2024-01-01 08:00:01,禮物卡,調味茶,檸檬茶,L,1,110,110
3,T00000004,3099,2024-01-01 08:00:04,現金,純咖啡,冷萃咖啡,L,1,140,140
4,T00000005,13482,2024-01-01 08:00:05,禮物卡,調味茶,櫻桃紅茶,L,1,120,120


In [5]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3081639 entries, 0 to 3081638
Data columns (total 10 columns):
 #   Column             Dtype 
---  ------             ----- 
 0   transaction_id     object
 1   member_id          int64 
 2   purchase_datetime  object
 3   payment_method     object
 4   category           object
 5   product            object
 6   size               object
 7   quantity           int64 
 8   unit_price         int64 
 9   total_price        int64 
dtypes: int64(4), object(6)
memory usage: 235.1+ MB


In [6]:
sales['quantity'].value_counts(normalize=True)

quantity
1     0.923592
2     0.057795
0     0.006812
10    0.001102
13    0.001101
19    0.001091
14    0.001082
15    0.001079
20    0.001061
11    0.001061
16    0.001061
17    0.001058
12    0.001057
18    0.001048
Name: proportion, dtype: float64

In [7]:
sales = sales[sales['quantity'] == 1]
sales['quantity'].value_counts()

quantity
1    2846178
Name: count, dtype: int64

In [8]:
sales['purchase_datetime'] = pd.to_datetime(sales['purchase_datetime'])
sales.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2846178 entries, 1 to 3081638
Data columns (total 10 columns):
 #   Column             Dtype         
---  ------             -----         
 0   transaction_id     object        
 1   member_id          int64         
 2   purchase_datetime  datetime64[ns]
 3   payment_method     object        
 4   category           object        
 5   product            object        
 6   size               object        
 7   quantity           int64         
 8   unit_price         int64         
 9   total_price        int64         
dtypes: datetime64[ns](1), int64(4), object(5)
memory usage: 238.9+ MB


In [17]:
sales.drop(columns=['quantity', 'total_price'], inplace=True)
sales = sales.rename(columns={'unit_price':'price'})

sales.head()

,transaction_id,member_id,purchase_datetime,payment_method,category,product,size,price
1,T00000002,3428,2024-01-01 08:00:01,電子支付,調味茶,櫻桃紅茶,L,120
2,T00000003,3428,2024-01-01 08:00:01,禮物卡,調味茶,檸檬茶,L,110
3,T00000004,3099,2024-01-01 08:00:04,現金,純咖啡,冷萃咖啡,L,140
4,T00000005,13482,2024-01-01 08:00:05,禮物卡,調味茶,櫻桃紅茶,L,120
5,T00000006,10752,2024-01-01 08:00:10,電子支付,奶茶,抹茶鮮奶,L,140


### Check duplicate and missing value

In [9]:
sales[sales.duplicated()]

,transaction_id,member_id,purchase_datetime,payment_method,category,product,size,quantity,unit_price,total_price


In [10]:
sales.isnull().sum()

transaction_id       0
member_id            0
purchase_datetime    0
payment_method       0
category             0
product              0
size                 0
quantity             0
unit_price           0
total_price          0
dtype: int64

## Member data

In [3]:
members = pd.read_csv('../raw/members.csv')
members.head()

,member_id,name,gender,register_date,birthday,card_level,CouponType
0,1,Jason Cheng,男,2020-01-30,1943-05-30,金,U
1,2,郭子瑜,女,2019-03-17,2004-06-30,金,U
2,3,賴芷嫣,女,2023-11-30,2001-11-23,銀,U
3,4,Emily Liu,女,2022-09-27,1976-02-26,鑽石,U
4,5,Linda Susan Wang,女,2024-05-01,2000-04-19,金,R


In [11]:
members.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   member_id      20000 non-null  int64 
 1   name           20000 non-null  object
 2   gender         20000 non-null  object
 3   register_date  20000 non-null  object
 4   birthday       20000 non-null  object
 5   card_level     20000 non-null  object
 6   CouponType     20000 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.1+ MB


In [12]:
members.drop(columns=['CouponType'], inplace=True)

In [13]:
members['register_date'] = pd.to_datetime(members['register_date'])
members['birthday'] = pd.to_datetime(members['birthday'])

members.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   member_id      20000 non-null  int64         
 1   name           20000 non-null  object        
 2   gender         20000 non-null  object        
 3   register_date  20000 non-null  datetime64[ns]
 4   birthday       20000 non-null  datetime64[ns]
 5   card_level     20000 non-null  object        
dtypes: datetime64[ns](2), int64(1), object(3)
memory usage: 937.6+ KB


In [14]:
members['age'] = 2026 - members['birthday'].dt.year
members['register_year'] = 2026 - members['register_date'].dt.year

members.head()

,member_id,name,gender,register_date,birthday,card_level,age,register_year
0,1,Jason Cheng,男,2020-01-30,1943-05-30,金,83,6
1,2,郭子瑜,女,2019-03-17,2004-06-30,金,22,7
2,3,賴芷嫣,女,2023-11-30,2001-11-23,銀,25,3
3,4,Emily Liu,女,2022-09-27,1976-02-26,鑽石,50,4
4,5,Linda Susan Wang,女,2024-05-01,2000-04-19,金,26,2


### Check duplicate and missing value

In [21]:
members[members.duplicated()]

,member_id,name,gender,register_date,birthday,card_level,age


In [22]:
members.isnull().sum()

member_id        0
name             0
gender           0
register_date    0
birthday         0
card_level       0
age              0
dtype: int64

## Merge two datasets

In [18]:
merge = pd.merge(sales, members, on='member_id', how='inner')
merge.head()

,transaction_id,member_id,purchase_datetime,payment_method,category,product,size,price,name,gender,register_date,birthday,card_level,age,register_year
0,T00000002,3428,2024-01-01 08:00:01,電子支付,調味茶,櫻桃紅茶,L,120,Michael Hsiao,男,2023-07-22,1950-11-19,鑽石,76,3
1,T00000003,3428,2024-01-01 08:00:01,禮物卡,調味茶,檸檬茶,L,110,Michael Hsiao,男,2023-07-22,1950-11-19,鑽石,76,3
2,T00000004,3099,2024-01-01 08:00:04,現金,純咖啡,冷萃咖啡,L,140,Eric James Wang,男,2021-03-01,1982-05-31,金,44,5
3,T00000005,13482,2024-01-01 08:00:05,禮物卡,調味茶,櫻桃紅茶,L,120,何佳穎,女,2022-01-15,1952-12-15,鑽石,74,4
4,T00000006,10752,2024-01-01 08:00:10,電子支付,奶茶,抹茶鮮奶,L,140,鄭雅婷,女,2022-01-26,2002-03-27,金,24,4


In [19]:
sales.to_csv('../clean/sales.csv', index=False)
members.to_csv('../clean/members.csv', index=False)
merge.to_csv('../clean/merge.csv', index=False)